# Crypto Price Prediction — All-in-one Colab notebook (Ripple / XRP)

**Version 5 — XRP-USD.** Same pipeline as v3 (baselines, Lag+Ridge + volume/volatility, LSTM returns + scaling + volume/volatility) but data is **Ripple (XRP)** instead of Bitcoin. Compare Dir.Acc and MAE/RMSE to BTC and ETH.

**Upload this file to Google Colab (File → Upload notebook) and run all cells.** No git, no clone, no setup. Optional: Runtime → Change runtime type → GPU for faster LSTM.

In [1]:
# Run this cell first: install packages (takes ~1 min)
!pip install -q pandas numpy yfinance pyarrow scikit-learn tensorflow matplotlib

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
import matplotlib.pyplot as plt

def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    n = len(y_true)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    if n > 1:
        true_dir = np.sign(np.diff(y_true))
        pred_dir = np.sign(y_pred[1:] - y_true[:-1])
        dir_acc = np.mean(true_dir == pred_dir)
    else:
        dir_acc = np.nan
    return {"mae": float(mae), "rmse": float(rmse), "directional_accuracy": float(dir_acc)}

# Data folder: Colab uses /content; local uses .
DATA_DIR = Path('/content/data') if Path('/content').exists() else Path('./data')
DATA_DIR.mkdir(exist_ok=True)
print("Ready. DATA_DIR =", DATA_DIR)

Ready. DATA_DIR = /content/data


---
## 1. Download data and split (70 / 15 / 15)

In [3]:
cache_path = DATA_DIR / "XRP_USD_daily.parquet"
if cache_path.exists():
    df = pd.read_parquet(cache_path)
    print("Loaded from cache:", cache_path)
else:
    raw = yf.download("XRP-USD", start="2017-01-01", end=None, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index().ffill().dropna()
    df = raw[["Close"]].copy()
    df.columns = ["price"]
    if "Volume" in raw.columns:
        df["volume"] = raw["Volume"]
    df.to_parquet(cache_path)
    print("Downloaded and saved:", cache_path)

# Ensure volume exists (use 0 if missing)
if "volume" not in df.columns:
    df["volume"] = 0.0
# Returns (day-over-day): NaN at first row
df["ret"] = (df["price"] - df["price"].shift(1)) / (df["price"].shift(1) + 1e-12)
# Rolling volatility: 14-day std of returns (past only via .rolling)
df["volatility_14"] = df["ret"].rolling(14).std()
# Log volume for scale
df["log_volume"] = np.log1p(df["volume"])
# Drop leading rows where volatility is NaN (optional; lag/LSTM will align)
df = df.copy()

n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]
print(df.shape, "| Train", len(train_df), "Val", len(val_df), "Test", len(test_df))

Downloaded and saved: /content/data/XRP_USD_daily.parquet
(3028, 5) | Train 2119 Val 454 Test 455


---
## 2. Baselines (last value, 7-day MA)

In [4]:
prices = test_df["price"].values
y_true = prices[1:]
pred_last = prices[:-1]
m_last = regression_metrics(y_true, pred_last)

window = 7
pred_ma = np.array([np.mean(prices[i - window : i]) for i in range(window, len(prices))])
y_true_ma = y_true[window - 1 :]
m_ma = regression_metrics(y_true_ma, pred_ma)

print("Last value:", m_last)
print("7-day MA:  ", m_ma)

Last value: {'mae': 0.07334290254483664, 'rmse': 0.11166687990108026, 'directional_accuracy': 0.0}
7-day MA:   {'mae': 0.12101191419119738, 'rmse': 0.17313756924770263, 'directional_accuracy': 0.5212527964205816}


---
## 3. Lag model (Ridge + 30 lags)

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

N_LAGS = 30
price_full = df["price"].values
log_vol_full = df["log_volume"].values
vol_full = np.nan_to_num(df["volatility_14"].values, nan=0.0)

def build_lag_features(price, n_lags):
    T = len(price)
    X_list = [price[n_lags - lag : T - lag] for lag in range(1, n_lags + 1)]
    X = np.column_stack(X_list)[:-1]
    y = price[n_lags + 1 :]
    return X, y

X_train, y_train = build_lag_features(train_df["price"].values, N_LAGS)
X_val, y_val = build_lag_features(val_df["price"].values, N_LAGS)
X_test, y_test = build_lag_features(test_df["price"].values, N_LAGS)

# Add volume and rolling volatility (aligned to last lag index per sample)
def add_vol_volatility(X, y, start_idx, n_lags):
    n = len(y)
    idx = start_idx + n_lags
    extra = np.column_stack([log_vol_full[idx : idx + n], vol_full[idx : idx + n]])
    return np.hstack([X, extra])

X_train = add_vol_volatility(X_train, y_train, 0, N_LAGS)
X_val   = add_vol_volatility(X_val,   y_val,   train_end, N_LAGS)
X_test  = add_vol_volatility(X_test,  y_test,  val_end, N_LAGS)

n_f = X_train.shape[1]
pipe = Pipeline([
    ("scale", ColumnTransformer([("s", StandardScaler(), list(range(n_f)))], remainder="passthrough")),
    ("ridge", Ridge(alpha=1.0)),
])
pipe.fit(X_train, y_train)
pred_lag = pipe.predict(X_test)
m_lag = regression_metrics(y_test, pred_lag)
print("Lag+Ridge (+ volume, volatility_14):", m_lag)

Lag+Ridge (+ volume, volatility_14): {'mae': 0.12808141779380317, 'rmse': 0.17175150809953085, 'directional_accuracy': 0.5153664302600472}


---
## 4. LSTM

In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 30
price = df["price"].values.astype(np.float64)
T = len(price)
# Returns: r[t] = (price[t+1] - price[t]) / price[t]
returns = (price[1:] - price[:-1]) / (price[:-1] + 1e-12)
returns = returns.astype(np.float32)
log_vol = df["log_volume"].values.astype(np.float32)
vol = np.nan_to_num(df["volatility_14"].values, nan=0.0).astype(np.float32)

def build_seq_multifeature(ret, log_vol, vol, start, end, seq_len):
    """Build sequences [ret, log_volume, volatility] per step; predict next return."""
    X_list = []
    y_list = []
    for i in range(start, min(end, len(ret) - 1)):
        if i >= seq_len:
            # Features at steps i-seq_len .. i-1 (all past)
            X_list.append(np.column_stack([
                ret[i - seq_len : i],
                log_vol[i - seq_len : i],
                vol[i - seq_len : i],
            ]))
            y_list.append(ret[i])
    if not X_list:
        return np.zeros((0, seq_len, 3), dtype=np.float32), np.array([], dtype=np.float32)
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y

X_tr, y_tr = build_seq_multifeature(returns, log_vol, vol, SEQ_LEN, train_end, SEQ_LEN)
X_va, y_va = build_seq_multifeature(returns, log_vol, vol, train_end, val_end, SEQ_LEN)
X_te, y_te = build_seq_multifeature(returns, log_vol, vol, val_end, T - 1, SEQ_LEN)

# Scale inputs (fit on train; shape n, seq_len, 3)
scaler = StandardScaler()
n_tr, seq_len, n_feat = X_tr.shape
X_tr_flat = X_tr.reshape(-1, n_feat)
scaler.fit(X_tr_flat)
X_tr = scaler.transform(X_tr_flat).reshape(-1, seq_len, n_feat).astype(np.float32)
X_va = scaler.transform(X_va.reshape(-1, n_feat)).reshape(-1, seq_len, n_feat).astype(np.float32)
X_te = scaler.transform(X_te.reshape(-1, n_feat)).reshape(-1, seq_len, n_feat).astype(np.float32)

model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, n_feat)),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(16),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=30, batch_size=32, verbose=0)

pred_ret = model.predict(X_te, verbose=0).ravel()
# Convert predicted return to price for comparison with other models (same test period)
test_start_price_idx = val_end
price_prev = price[test_start_price_idx : test_start_price_idx + len(pred_ret)]
pred_lstm_price = price_prev * (1 + pred_ret)
y_true_price = price[test_start_price_idx + 1 : test_start_price_idx + 1 + len(pred_ret)]
m_lstm = regression_metrics(y_true_price, pred_lstm_price)
print("LSTM (+ volume, volatility_14, predict returns → price):", m_lstm)

LSTM (+ volume, volatility_14, predict returns → price): {'mae': 0.0892744322095423, 'rmse': 0.12804589118919835, 'directional_accuracy': 0.5}


---
## 5. Comparison table

In [7]:
rows = [
    ["Last value", m_last["mae"], m_last["rmse"], m_last["directional_accuracy"]],
    ["7-day MA", m_ma["mae"], m_ma["rmse"], m_ma["directional_accuracy"]],
    ["Lag+Ridge", m_lag["mae"], m_lag["rmse"], m_lag["directional_accuracy"]],
    ["LSTM", m_lstm["mae"], m_lstm["rmse"], m_lstm["directional_accuracy"]],
]
print(pd.DataFrame(rows, columns=["Model", "MAE", "RMSE", "Dir.Acc"]).to_string(index=False))

     Model      MAE     RMSE  Dir.Acc
Last value 0.073343 0.111667 0.000000
  7-day MA 0.121012 0.173138 0.521253
 Lag+Ridge 0.128081 0.171752 0.515366
      LSTM 0.089274 0.128046 0.500000
